<a href="https://colab.research.google.com/github/Smolry/Smart-HSRP-detection/blob/main/resnet_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
santoshvishwakarma99_indian_license_plate_dataset_path = kagglehub.dataset_download('santoshvishwakarma99/indian-license-plate-dataset')
santoshvishwakarma99_plate2_path = kagglehub.dataset_download('santoshvishwakarma99/plate2')
santoshvishwakarma99_plate3_n_path = kagglehub.dataset_download('santoshvishwakarma99/plate3-n')
santoshvishwakarma99_plates_path = kagglehub.dataset_download('santoshvishwakarma99/plates')
shivashankar2445_resnet_101_keras_default_1_path = kagglehub.model_download('shivashankar2445/resnet-101/Keras/default/1')
santoshvishwakarma99_resnet1_1_1_other_default_1_path = kagglehub.model_download('santoshvishwakarma99/resnet1.1_1/Other/default/1')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
import matplotlib.image as mpimg
import pandas as pd
from PIL import Image
from tqdm import tqdm
import seaborn as sns
import keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tensorflow.keras import models, optimizers, regularizers
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Activation, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:


# Define the classes (labels) for the HSRP license plate dataset
hsrp_labels = ['hsrp', 'non-hsrp']

# Define the paths to the train, validation, and test datasets
train_features_images = '/kaggle/input/indian-license-plate-dataset/dataset/train'
val_features_images = '/kaggle/input/indian-license-plate-dataset/dataset/val/'

# Note: Test dataset path is not defined in the dataset description. If available, add it similarly:
# test_features_images = '/kaggle/input/indian-license-plate-dataset/dataset/test/'



In [ ]:
# Model and training parameters
base_filters = 32  # Number of filters for the base convolutional layer
lrate = 0.001  # Learning rate
l1 = 0.  # L1 regularization value
l2 = 0.  # L2 regularization value
w_regularizers = 1e-5  # Weight regularization value
regularizer = tf.keras.regularizers.l1_l2(l1, l2)

# Image parameters
IMG_HEIGHT = 512  # Height of input images
IMG_WIDTH = 512  # Width of input images
image_size = (IMG_HEIGHT, IMG_WIDTH)

# Batch size for training
batch_size = 256

# Number of channels in the images (e.g., 3 for RGB)
NUM_CHANNELS = 3


In [ ]:
# Load the training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_features_images,
    labels='inferred',
    label_mode='categorical',  # Outputs labels as one-hot encoded categories
    seed=13,                   # Seed for reproducibility
    image_size=image_size,     # Resize images to the specified size
    batch_size=batch_size,     # Batch size for training
)

# Load the validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_features_images,
    labels='inferred',
    label_mode='categorical',  # Outputs labels as one-hot encoded categories
    seed=13,                   # Seed for reproducibility
    image_size=image_size,     # Resize images to the specified size
    batch_size=batch_size,     # Batch size for validation
)


In [ ]:
train_ds

In [ ]:
val_ds

In [ ]:
import matplotlib.pyplot as plt

# Visualize a few samples from the training dataset
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(hsrp_labels[np.argmax(labels[i])])  # Use HSRP labels instead of animal labels
        plt.axis("off")



In [ ]:
import tensorflow as tf

# Define a data augmentation pipeline
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(
        mode='horizontal',  # Randomly flips the image horizontally
        name='random_lr_flip_none'  # Updated name
    )
])


In [ ]:
# Apply `data_aug` to the training images
train_ds = train_ds.map(
    lambda img, label: (data_aug(img), label),  # Apply data augmentation only to images
    num_parallel_calls=tf.data.AUTOTUNE        # Optimize the data pipeline
)


In [ ]:
# Prefetching samples in GPU memory to maximize GPU utilization
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)  # Prefetch training dataset
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)      # Prefetch validation dataset


In [ ]:
train_ds

In [ ]:
val_ds

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.optimizers import Adam

# Define the variables
IMG_HEIGHT = 512
IMG_WIDTH = 512
num_classes = 2  # Number of output classes (hsrp and non-hsrp)

# Load the pre-trained ResNet-101 model (excluding the top layer)
base_model = ResNet101(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

# Freeze the layers of the pre-trained model
for layer in base_model.layers:
    if not isinstance(layer, layers.BatchNormalization):  # Leave BatchNorm layers trainable
        layer.trainable = False

# Build the model using the Functional API for better flexibility
inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model(inputs, training=False)  # Ensure base_model stays in inference mode
x = layers.GlobalAveragePooling2D()(x)  # Global Average Pooling layer

# Fully connected layers
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

# Output layer for classification (hsrp vs non-hsrp)
outputs = layers.Dense(num_classes, activation='softmax')(x)

# Create the final model
model = models.Model(inputs, outputs)

# Compile the model with fine-tuning adjustments
model.compile(
    optimizer=Adam(learning_rate=1e-4),  # Smaller initial learning rate for better convergence
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print the model summary
model.summary()

# Save the final retrained model
model.save('/kaggle/working/resnet101_model_hsrp.h5')
print("Final retrained HSRP model saved successfully!")


In [ ]:
# Get a single batch from the validation dataset
images, labels = next(iter(val_ds))

# Optionally, visualize the first image in the batch
plt.imshow(images[0].numpy().astype("uint8"))
plt.title(f"Label: {hsrp_labels[np.argmax(labels[0])]}")  # Display the label for the first image
plt.axis('off')
plt.show()


In [ ]:
# Get a single batch from the training dataset
images, labels = next(iter(train_ds))

# Optionally, visualize the first image in the batch
plt.imshow(images[0].numpy().astype("uint8"))
plt.title(f"Label: {hsrp_labels[np.argmax(labels[0])]}")  # Display the label for the first image
plt.axis('off')
plt.show()


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Hyperparameters
IMG_HEIGHT = 512  # Reduced resolution
IMG_WIDTH = 512
image_size = (IMG_HEIGHT, IMG_WIDTH)
batch_size = 8  # Reduced batch size
lrate = 0.001
epochs = 30

# Paths to datasets
train_data_dir = '/kaggle/input/indian-license-plate-dataset/dataset/train'
validation_data_dir = '/kaggle/input/indian-license-plate-dataset/dataset/val'

# Load the pre-trained model
model_path = '/kaggle/working/resnet101_model_hsrp.h5'  # Adjust the model path
try:
    model = load_model(model_path)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# Data augmentation configuration
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='sparse',  # We have sparse labels (hsrp and non-hsrp)
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    validation_data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='sparse',  # We have sparse labels (hsrp and non-hsrp)
    shuffle=False
)

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lrate),
    loss='sparse_categorical_crossentropy',  # Suitable for sparse labels
    metrics=['accuracy']
)

# Add EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Add TensorBoard callback for visualization
log_dir = os.path.join("/kaggle/working/logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Train the model
print("Starting training...")
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=[early_stopping, tensorboard_callback],
    verbose=1
)

# Save the retrained model
retrained_model_path = '/kaggle/working/my_resnet101_hsrp_model.h5'
model.save(retrained_model_path)
print(f"Retrained model saved to: {retrained_model_path}")


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Paths
model_path = "/kaggle/working/my_resnet101_hsrp_model.h5"
val_dataset_path = "/kaggle/input/indian-license-plate-dataset/dataset/train"

# Load the pre-trained model
model = load_model(model_path)

# Check model summary and expected input shape
print("Model Summary:")
model.summary()
expected_input_shape = model.input_shape[1:3]  # Extract expected height and width

# Data generator for the validation dataset
datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)

# Ensure validation generator uses the correct input shape
val_generator = datagen.flow_from_directory(
    val_dataset_path,
    target_size=expected_input_shape,  # Match model's expected input shape
    batch_size=32,  # Use a batch size of 32
    class_mode="categorical",  # Model expects categorical (one-hot encoded) labels
    shuffle=False  # Maintain order for consistency in predictions and ground truth
)

# Predict on the validation dataset
try:
    # Ground truth labels and predictions
    y_true = val_generator.classes  # True labels from the validation generator
    y_pred_prob = model.predict(val_generator, verbose=1)  # Predicted probabilities
    y_pred = np.argmax(y_pred_prob, axis=1)  # Convert probabilities to class predictions
except ValueError as e:
    print("Error during prediction:", e)
    raise

# Get class names from the validation generator
class_names = list(val_generator.class_indices.keys())

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(12, 8))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "Count"},
)
plt.title("Confusion Matrix", fontsize=16)
plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Classification report
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

# Convert the classification report to a DataFrame for better readability
report_df = pd.DataFrame(report).transpose()

# Print the classification report
print("Classification Report:")
print(report_df)

# Save the classification metrics to a CSV file
report_df.to_csv("classification_metrics_val.csv", index=True)

# Extract class-wise metrics
accuracy = accuracy_score(y_true, y_pred)
class_wise_metrics = report_df.loc[class_names, ["precision", "recall", "f1-score"]]
class_wise_metrics["support"] = report_df.loc[class_names, "support"]

# Create class-wise metrics table
accuracy_table = pd.DataFrame({
    "Class": class_names,
    "Precision": class_wise_metrics["precision"],
    "Recall": class_wise_metrics["recall"],
    "F1-Score": class_wise_metrics["f1-score"],
    "Support": class_wise_metrics["support"]
})
print("\nClass-Wise Metrics Table:")
print(accuracy_table)

# Save the accuracy table as a CSV file
accuracy_table.to_csv("class_wise_metrics_val.csv", index=False)

# Print overall accuracy
print(f"\nOverall Accuracy: {accuracy * 100:.2f}%")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Paths
model_path = "/kaggle/working/my_resnet101_hsrp_model.h5"
val_dataset_path = "/kaggle/input/indian-license-plate-dataset/dataset/val"

# Load the pre-trained model
model = load_model(model_path)

# Check model summary and expected input shape
print("Model Summary:")
model.summary()
expected_input_shape = model.input_shape[1:3]  # Extract expected height and width

# Data generator for the validation dataset
datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)

# Ensure validation generator uses the correct input shape
val_generator = datagen.flow_from_directory(
    val_dataset_path,
    target_size=expected_input_shape,  # Match model's expected input shape
    batch_size=32,  # Use a batch size of 32
    class_mode="categorical",  # Model expects categorical (one-hot encoded) labels
    shuffle=False  # Maintain order for consistency in predictions and ground truth
)

# Predict on the validation dataset
try:
    # Ground truth labels and predictions
    y_true = val_generator.classes  # True labels from the validation generator
    y_pred_prob = model.predict(val_generator, verbose=1)  # Predicted probabilities
    y_pred = np.argmax(y_pred_prob, axis=1)  # Convert probabilities to class predictions
except ValueError as e:
    print("Error during prediction:", e)
    raise

# Get class names from the validation generator
class_names = list(val_generator.class_indices.keys())

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(12, 8))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "Count"},
)
plt.title("Confusion Matrix", fontsize=16)
plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Classification report
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

# Convert the classification report to a DataFrame for better readability
report_df = pd.DataFrame(report).transpose()

# Print the classification report
print("Classification Report:")
print(report_df)

# Save the classification metrics to a CSV file
report_df.to_csv("classification_metrics_val.csv", index=True)

# Extract class-wise metrics
accuracy = accuracy_score(y_true, y_pred)
class_wise_metrics = report_df.loc[class_names, ["precision", "recall", "f1-score"]]
class_wise_metrics["support"] = report_df.loc[class_names, "support"]

# Create class-wise metrics table
accuracy_table = pd.DataFrame({
    "Class": class_names,
    "Precision": class_wise_metrics["precision"],
    "Recall": class_wise_metrics["recall"],
    "F1-Score": class_wise_metrics["f1-score"],
    "Support": class_wise_metrics["support"]
})
print("\nClass-Wise Metrics Table:")
print(accuracy_table)

# Save the accuracy table as a CSV file
accuracy_table.to_csv("class_wise_metrics_val.csv", index=False)

# Print overall accuracy
print(f"\nOverall Accuracy: {accuracy * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Retrieve the training history
history_dict = history.history

# Plot Accuracy
plt.figure(figsize=(12, 6))

# Plot training and validation accuracy
plt.subplot(1, 2, 1)
plt.plot(history_dict['accuracy'], label='Training Accuracy')
plt.plot(history_dict['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history_dict['loss'], label='Training Loss')
plt.plot(history_dict['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Show the plots
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('/kaggle/working/my_resnet101_hsrp_model.h5')  # Adjust the path to your saved model

# Define class labels (adjust based on your dataset)
class_labels = ['hsrp','non-hsrp']

# Function to preprocess and predict an image
def predict_and_display(image_path, model, target_size=(512, 512), class_labels=None):
    # Load and preprocess the image (resize to the correct input size for the model)
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0  # Normalize the image
    img_batch = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make a prediction
    predictions = model.predict(img_batch)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class = class_labels[predicted_class_index] if class_labels else predicted_class_index
    confidence_score = predictions[0][predicted_class_index]

    # Display the image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class} ({confidence_score:.2f})")
    plt.show()

# Provide the path to your input image
image_path = '/kaggle/input/indian-license-plate-dataset/dataset/val/hsrp/0CKYHOL_lgmnkp83079.jpg'  # Replace with your image path

# Predict and display the result
predict_and_display(image_path, model, target_size=(512, 512), class_labels=class_labels)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('/kaggle/working/my_resnet101_hsrp_model.h5')  # Adjust the path to your saved model

# Define class labels (adjust based on your dataset)
class_labels = ['hsrp', 'non-hsrp']

# Function to preprocess and predict an image
def predict_and_display(image_path, model, target_size=(512, 512), class_labels=None):
    # Load and preprocess the image (resize to the correct input size for the model)
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0  # Normalize the image
    img_batch = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make a prediction
    predictions = model.predict(img_batch)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class = class_labels[predicted_class_index] if class_labels else predicted_class_index
    confidence_score = predictions[0][predicted_class_index]

    # Display the image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class} ({confidence_score:.2f})")
    plt.show()

# Provide the path to your input image
image_path = '/kaggle/input/plate2/plate2.jpeg'  # Corrected path with raw string

# Predict and display the result
predict_and_display(image_path, model, target_size=(512, 512), class_labels=class_labels)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('/kaggle/working/my_resnet101_hsrp_model.h5')  # Adjust the path to your saved model

# Define class labels (adjust based on your dataset)
class_labels = ['hsrp', 'non-hsrp']

# Function to preprocess and predict an image
def predict_and_display(image_path, model, target_size=(512, 512), class_labels=None):
    # Load and preprocess the image (resize to the correct input size for the model)
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0  # Normalize the image
    img_batch = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make a prediction
    predictions = model.predict(img_batch)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class = class_labels[predicted_class_index] if class_labels else predicted_class_index
    confidence_score = predictions[0][predicted_class_index]

    # Display the image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class} ({confidence_score:.2f})")
    plt.show()

# Provide the path to your input image
image_path = '/kaggle/input/plate3-n/plate5.jpeg'  # Corrected path with raw string

# Predict and display the result
predict_and_display(image_path, model, target_size=(512, 512), class_labels=class_labels)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('/kaggle/input/resnet1.1_1/other/default/1/my_resnet101_hsrp_model.h5')  # Adjust the path to your saved model

# Define class labels (adjust based on your dataset)
class_labels = ['hsrp', 'non-hsrp']

# Function to preprocess and predict an image
def predict_and_display(image_path, model, target_size=(512, 512), class_labels=None):
    # Load and preprocess the image (resize to the correct input size for the model)
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0  # Normalize the image
    img_batch = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make a prediction
    predictions = model.predict(img_batch)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class = class_labels[predicted_class_index] if class_labels else predicted_class_index
    confidence_score = predictions[0][predicted_class_index]

    # Display the image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class} ({confidence_score:.2f})")
    plt.show()

# Provide the path to your input image
image_path = '/kaggle/input/plates/OIP (1).jpeg'  # Corrected path with raw string

# Predict and display the result
predict_and_display(image_path, model, target_size=(512, 512), class_labels=class_labels)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('/kaggle/input/resnet1.1_1/other/default/1/my_resnet101_hsrp_model.h5')  # Adjust the path to your saved model

# Define class labels (adjust based on your dataset)
class_labels = ['hsrp', 'non-hsrp']

# Function to preprocess and predict an image
def predict_and_display(image_path, model, target_size=(512, 512), class_labels=None):
    # Load and preprocess the image (resize to the correct input size for the model)
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0  # Normalize the image
    img_batch = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make a prediction
    predictions = model.predict(img_batch)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class = class_labels[predicted_class_index] if class_labels else predicted_class_index
    confidence_score = predictions[0][predicted_class_index]

    # Display the image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class} ({confidence_score:.2f})")
    plt.show()

# Provide the path to your input image
image_path = '/kaggle/input/plates/OIP.jpeg'  # Corrected path with raw string

# Predict and display the result
predict_and_display(image_path, model, target_size=(512, 512), class_labels=class_labels)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('/kaggle/input/resnet1.1_1/other/default/1/my_resnet101_hsrp_model.h5')  # Adjust the path to your saved model

# Define class labels (adjust based on your dataset)
class_labels = ['hsrp', 'non-hsrp']

# Function to preprocess and predict an image
def predict_and_display(image_path, model, target_size=(512, 512), class_labels=None):
    # Load and preprocess the image (resize to the correct input size for the model)
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0  # Normalize the image
    img_batch = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make a prediction
    predictions = model.predict(img_batch)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class = class_labels[predicted_class_index] if class_labels else predicted_class_index
    confidence_score = predictions[0][predicted_class_index]

    # Display the image with prediction
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class} ({confidence_score:.2f})")
    plt.show()

# Provide the path to your input image
image_path = '/kaggle/input/plates/download.jpeg'  # Corrected path with raw string

# Predict and display the result
predict_and_display(image_path, model, target_size=(512, 512), class_labels=class_labels)
